# Make Hot Star Subsamples
## version history
- last update 2026-06-27 by Miller and Hogg

In [ ]:
from pathlib import Path
import sys
import pickle
import hashlib
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt

In [ ]:
#set up to use hpc
%matplotlib widget

sys.path.append('/nexus/posix0/MIA-astro-env/hxr/bepennell/emission_line_stars/HotStarsBOSS/code/')
sys.path.append('../code/')
from spec_utils import load_first_spectrum

# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

In [ ]:
# Create dictionary of disk locations on the hpc
CONFIG = {
    'project_root': '/nexus/posix0/MIA-astro-env/hxr/bepennell/emission_line_stars/HotStarsBOSS',
    'merged_star_file': '/nexus/posix0/MIA-astro-env/hxr/shared/FEROS_data/BOSS_data/info/allStar-v6_2_1_merged_params.parquet', 
    'merged_spectra_file': '/nexus/posix0/MIA-astro-env/hxr/shared/FEROS_data/BOSS_data/info/spAll-v6_2_1_merged_params.parquet',
    'bossnet_file': Path('/nexus/posix0/MIA-astro-env/hxr/shared/FEROS_data/BOSS_data/bossnet/astraAllStarBossNet_filtered.parquet'),
    'raw_cache_dir': Path('/nexus/posix0/MIA-astro-env/hxr/shared/FEROS_data/BOSS_data/boss_v6_2_1_cache'),
    'normalised_dir': Path('/nexus/posix0/MIA-astro-env/hxr/shared/FEROS_data/BOSS_data/boss_v6_2_1_normalised'),
}

# CONFIG['output_dir'].mkdir(parents=True, exist_ok=True)
# CONFIG['figures_dir'].mkdir(parents=True, exist_ok=True)

In [ ]:
#read table of stars in spectra
stars = pd.read_parquet(CONFIG['merged_star_file']) 
spectra = pd.read_parquet(CONFIG['merged_spectra_file'])

In [ ]:
# TODO: We need to choose a PARENT SAMPLE of STARS.
# WE need also to get the housekeeping data aligned with the parent sample.
# We also need to tag them with something that can be used to do A/B split
#about the text so that it's consistent with this code

In [ ]:
#masking for various relevant quantites
mask_quality_stars = (stars['is_quality_star'] == True)
mask_hot_stars = (stars['Teff_fit'] >= 15000) # make a histo to see distribution
mask_no_emission = stars['EWobs_Halpha_mean'] >0.

In [ ]:
# The cold stars indices and emisison star indices are only used for the plotting, not for 
cold_star_inds = stars.loc[mask_quality_stars & ~mask_hot_stars].index
emission_star_inds = stars.loc[mask_quality_stars & ~mask_no_emission].index

print(f'Quality filtering for stars: kept {len(stars.loc[mask_quality_stars & mask_hot_stars & mask_no_emission])} / {len(stars)} rows')
print(f"Stars with Teff_fit >= 10000: {len(stars.loc[mask_hot_stars])} / {len(stars)} rows")
print(f"Stars with EWobs_Halpha_mean > 0: {len(stars.loc[mask_no_emission])} / {len(stars)} rows")
print(f'Quality filtering for stars: kept {len(stars.loc[mask_quality_stars])} / {len(stars)} rows')

In [ ]:
# make the parent sample of stars
parent_stars = stars.loc[mask_quality_stars & mask_hot_stars]
print("we got", len(parent_stars), "stars in the parent sample of stars")

In [ ]:
# make the housekeeping data in which we will do the A B split
# the next line will fail for all sorts of reasons; one is that we have to binary-encode "foo".
parent_stars.insert(loc = 1, column = "NANA_HASH", value = -1)
parent_stars['NANA_HASH'] = [int(hashlib.md5(foo).hexdigest(), 16) % 3570 for q, foo in stars['SDSS_ID']]
print(stars['NANA_HASH'])

In [ ]:
# write out parquet files
stars_fn = "parent_stars_2026-07-08.parquet"
parent_stars.to_parquet(stars_fn)

In [ ]:
# in case you want it, let's choose a small sample of only 2000 stars
# if you want to run ALL, set subsample_size = None.
subsample_size = 2000 # None
parent_stars = pd.read_parquet(stars_fn)
if subsample_size is not None:
    rng = np.random.default_rng(17)
    subset_index = np.sort(rng.choice(np.arange(len(parent_stars)), size=subsample_size, replace=False))
    parent_stars = parent_stars.iloc[subset_index]
print(len(parent_stars))

In [ ]:
# reminder of how to force ipynb to list all column names
print(parent_spectra.columns.to_list())

In [ ]:
# BUG: THIS IS WRONG
# We want ALL spectra of the parent_stars
# do this correctly, like this pseudo-code here

# make the parent sample of spectra
# I want to do the following; NANA will figure out how to do it.
# In the documentation, this might be called an "outer join" or a "left join".
parent_spectra = spectra[spectra['SDSS_ID'] in parent_stars['SDSS_ID']]
print("we got", len(parent_spectra), "spectra of stars in the parent sample of stars")

In [ ]:
#read and package one spectrum from each star in the small subsample
normalized_dir=CONFIG['normalised_dir']

N = len(spec_files)
M = None
for ii, spec_file in enumerate(spec_files):
        
    plate = spec_file.split('-')[1] 
    key = spec_file.replace('.fits', '')  
    with h5py.File(normalized_dir / f'{plate}.h5', 'r') as f: 
            
            flux1 = f[key]['FLUX'][:]        
            loglam1 = f[key]['LOGLAM'][:]   
            ivar1 = f[key]['IVAR'][:]        
            continuum1 = f[key]['CONTINUUM'][:]

            if M is None:
                M = len(flux1)
                flux = np.ones((N, M))
                loglam = loglam1.copy()
                ivar = np.zeros((N, M))
                continuum = np.zeros((N, M))
            assert len(flux1) == M
            assert np.allclose(loglam1, loglam)

            flux[ii] = flux1
            ivar[ii] = ivar1
            continuum[ii] = continuum1
            spec_files[ii] = spec_file

print(flux.shape, loglam.shape, ivar.shape, continuum.shape)

In [ ]:
#save hot stars sample

file_prefix = "hot_stars_subsample_2026-06-27"
hot_stars_subsample.to_parquet(file_prefix + "_table.parquet")

In [ ]:
# how many failures did we have?
print(hot_stars_subsample[hot_stars_subsample["successfully_read_file"] == False])

In [ ]:
# spot check
plt.figure()
plt.plot(flux[1995])
plt.show()

In [ ]:
# write a pickle file
with open(file_prefix + "_data.pkl", "wb") as file:
    pickle.dump((flux, loglam, ivar, continuum, spec_files, spectra), file)